In [2]:
import jax.numpy as jnp
from jax import grad, jit, vmap
from jax import random
from jax import lax
from optimization import optimize_fire2
from potentials import effective_potential, total_effective_potential
import numpy as onp
from matplotlib import pyplot as plt

from example_many_rods import create_random_rods

In [2]:
def forward3(w):
    def body(carry, w):
        conv = jnp.convolve(carry * w, kernel, mode='valid')
        out = jnp.zeros_like(w).at[1:-1].set(conv)
        return out, out
    
    init = jnp.ones(w.shape[0])
    kernel = jnp.ones(3)
    return jnp.vstack([init, lax.scan(body, jnp.ones(w.shape[0]), w.T)[1]]).T

In [11]:
width = 32
height = 64
key = random.key(0)
w = random.uniform(key, (height, width))

# p1s = random.uniform(key, (num_rods,3))



In [12]:
%time forward3(w).block_until_ready()
# CPU times: user 93.2 ms, sys: 2.96 ms, total: 96.1 ms
# Wall time: 94 ms

CPU times: user 99.1 ms, sys: 4.38 ms, total: 103 ms
Wall time: 170 ms


Array([[1.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [1.0000000e+00, 5.3757513e-01, 3.6503273e-01, ..., 2.6597607e+04,
        4.0904156e+04, 3.5301363e+04],
       [1.0000000e+00, 1.0154681e+00, 3.9626214e-01, ..., 7.6378648e+04,
        4.9757328e+04, 1.2892481e+05],
       ...,
       [1.0000000e+00, 1.3469143e+00, 1.6578368e+00, ..., 2.4548130e+03,
        7.6722349e+03, 8.8550264e+03],
       [1.0000000e+00, 1.4892446e+00, 1.4656001e+00, ..., 7.2333270e+02,
        1.8449697e+03, 5.1617441e+03],
       [1.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00]], dtype=float32)

In [14]:
rod_length = 1.0
dist = 0.5

num_rods = 10
# create jnp random array
key = random.key(0)
p1s = random.uniform(key, (num_rods,3))
key = random.key(1)
phi1 = random.uniform(key, (num_rods,1), minval=0., maxval=jnp.pi)
key = random.key(2)
theta1 = random.uniform(key, (num_rods,1), minval=0., maxval=2*jnp.pi)

# key = random.key(3)
# p2s = random.normal(key, (num_rods,3))
# key = random.key(4)
# phi2 = random.uniform(key, (num_rods,1), minval=0., maxval=jnp.pi)
# key = random.key(5)
# theta2 = random.uniform(key, (num_rods,1), minval=0., maxval=2*jnp.pi)

q0 = jnp.concatenate([p1s, phi1, theta1], axis=1)
# q0 = q0.flatten()
print(q0.shape)

from potentials import total_effective_potential


%time total_effective_potential(q0).block_until_ready()

(10, 5)
CPU times: user 159 ms, sys: 4 ms, total: 163 ms
Wall time: 156 ms


Array(13.995945, dtype=float32)

In [15]:
fval = total_effective_potential(q0)
print(f"f_val: {fval:.2f}")

f_val: 14.00


In [16]:

total1 = 0
k = 0
for i in range(N):
    q_i = q0[i]
    for j in range(i+1, N):
        q_j = q0[j]
        # q_pairs[k,:] = jnp.concatenate([q_i, q_j])
        q_pairs = q_pairs.at[k].set(jnp.concatenate([q_i, q_j]))
        total1 += effective_potential(jnp.concatenate([q_i, q_j]))
        k += 1                        
    
end_time = time.time()
execution_time = end_time - start_time
print(f"Execution time (1): {execution_time:.2f} seconds")


start_time = time.time()
total2 = total_effective_potential(q_pairs)
end_time = time.time()
execution_time = end_time - start_time
print(f"Execution time (2): {execution_time:.2f} seconds")


print(total1)
print(total2)

NameError: name 'N' is not defined

In [35]:
from potentials import compute_linking_number
import numpy as np

key = random.key(0)
q = random.uniform(key, (10,))

print(q)

q = jnp.array([-5,0,0,jnp.pi/2,0,
               0,-5,0.2,jnp.pi/2,jnp.pi/2,])

compute_linking_number(*q,10)

for i in range(10000,20000):
    key = random.key(i)
    q = random.uniform(key, (10,))    

    # Check if nan occurs
    if np.isnan(q).any():
        print("nan occurred in q")    



[0.35490513 0.60419905 0.4275843  0.23061597 0.32985854 0.43953657
 0.25099766 0.27730572 0.7678207  0.71474564]


In [53]:
q = jnp.array([0.56577194, 0.2576256, 0.87928945, 1.383639, 1.0297008,0.8941607,0.59665227, 0.4533488,  3.0993779,  4.7968893 ])

effective_potential(q)

Array(0.17312227, dtype=float32)

In [58]:
# q_pair = jnp.array([[0.56577194, 0.2576256, 0.87928945, 1.383639, 1.0297008],
#                     [0.8941607,0.59665227, 0.4533488,  3.0993779,  4.7968893]])
print(jnp.reshape(q,(1,10)))

# repeat q 10 times
q = jnp.array([-5,0,0,jnp.pi/2,0,
               0,-5,0.2,jnp.pi/2,jnp.pi/2,])
q_pairs = jnp.repeat(jnp.reshape(q,(1,10)), 10, axis=0)
print(q_pairs)

print(total_effective_potential(q_pairs)/10)


[[0.56577194 0.2576256  0.87928945 1.383639   1.0297008  0.8941607
  0.59665227 0.4533488  3.0993779  4.7968893 ]]
[[0.56577194 0.2576256  0.87928945 1.383639   1.0297008  0.8941607
  0.59665227 0.4533488  3.0993779  4.7968893 ]
 [0.56577194 0.2576256  0.87928945 1.383639   1.0297008  0.8941607
  0.59665227 0.4533488  3.0993779  4.7968893 ]
 [0.56577194 0.2576256  0.87928945 1.383639   1.0297008  0.8941607
  0.59665227 0.4533488  3.0993779  4.7968893 ]
 [0.56577194 0.2576256  0.87928945 1.383639   1.0297008  0.8941607
  0.59665227 0.4533488  3.0993779  4.7968893 ]
 [0.56577194 0.2576256  0.87928945 1.383639   1.0297008  0.8941607
  0.59665227 0.4533488  3.0993779  4.7968893 ]
 [0.56577194 0.2576256  0.87928945 1.383639   1.0297008  0.8941607
  0.59665227 0.4533488  3.0993779  4.7968893 ]
 [0.56577194 0.2576256  0.87928945 1.383639   1.0297008  0.8941607
  0.59665227 0.4533488  3.0993779  4.7968893 ]
 [0.56577194 0.2576256  0.87928945 1.383639   1.0297008  0.8941607
  0.59665227 0.45334

In [4]:
q0 = create_random_rods(10)
print(q0)

[[0.38831604 0.82557344 0.15744662 2.3723924  0.289822  ]
 [0.9216856  0.8541341  0.81880295 0.9832334  4.5968437 ]
 [0.33484733 0.83797956 0.28858185 0.38919222 3.3833423 ]
 [0.5061135  0.01882851 0.7104969  1.7221833  0.37405762]
 [0.8197037  0.71717155 0.7650944  1.3267298  1.3981359 ]
 [0.5481789  0.37306917 0.9477587  0.96060455 1.3749892 ]
 [0.59677327 0.08364284 0.27055883 2.576371   2.8958561 ]
 [0.5558171  0.7998272  0.136899   3.004423   4.5412493 ]
 [0.14062798 0.12721694 0.27647746 1.1203711  5.3149724 ]
 [0.43810904 0.35078037 0.13254273 1.7496059  2.7232983 ]]


In [10]:
tmp = (q0[jnp.newaxis,:] - q0[:,jnp.newaxis])
tmp.shape

(10, 10, 5)

In [5]:
q0 = create_random_rods(10)
N = 10

total1 = total_effective_potential(q0)
total2 = 0
for i in range(N):
    for j in range(i+1, N):
        q_i = q0[i]
        q_j = q0[j]
        q_pairs = jnp.concatenate([q_i, q_j])
        total2 += effective_potential(q_pairs)
        
print(total1)
print(total2)

-2.2788765
-2.2788765


In [3]:
q_opt = jnp.array([0.90764815, 0.13361165, 0.6639446,  1.7948972 , 4.1847196 , 0.7265267,
 0.32963756, 0.65795946, 1.0229627,  1.1714247 , 0.65857667, 0.02727637,
 0.45270422, 2.8017418 , 3.548705 ,  0.15392938, 0.26065966, 0.67541796,
 1.4439802 , 0.33963197])

total_effective_potential(q_opt)

Array(-1.2674792, dtype=float32)

In [4]:
q_reshaped = jnp.reshape(q_opt, (-1,5))
N = q_reshaped.shape[0]

for i in range(N):
    for j in range(i+1, N):
        q_i = q_reshaped[i]
        q_j = q_reshaped[j]
        q_pairs = jnp.concatenate([q_i, q_j])
        
        if jnp.isnan(effective_potential(q_pairs)):
            print(f"nan occurred at {i} and {j}")    
            print(q_i)
            print(q_j)
            
            q_in = q_i
            q_jn = q_j            
        

In [4]:
from potentials import compute_linking_number
q = jnp.concatenate([q_in,q_jn])
print(q)
x_i = q[0]
y_i = q[1]
z_i = q[2]
phi_i = q[3]
theta_i = q[4]

x_j = q[5]
y_j = q[6]
z_j = q[7]
phi_j = q[8]
theta_j = q[9]

p_i = jnp.array([x_i, y_i, z_i])
p_j = jnp.array([x_j, y_j, z_j])
u_i = jnp.array([jnp.sin(phi_i)*jnp.cos(theta_i), jnp.sin(phi_i)*jnp.sin(theta_i), jnp.cos(phi_i)])
u_j = jnp.array([jnp.sin(phi_j)*jnp.cos(theta_j), jnp.sin(phi_j)*jnp.sin(theta_j), jnp.cos(phi_j)])



compute_linking_number(x_i, y_i, z_i, phi_i, theta_i, x_j, y_j, z_j, phi_j, theta_j, 1)

NameError: name 'q_in' is not defined